# Open coagulation, conservative engine, constant kernel

Rebuilt from `runs/open_coagulation_conservative_constant.npz`.

В конец добавлены два PRL-рисунка из
`Analysis_open_coagulation_conservative_geometric.ipynb` — сырой $dN/dm$ с
изохронами и логнормальной подгонкой старшей из них, и однопанельный
компенсированный $m^2\,dN/dm$. Всё, что было зашито под геометрический
прогон — тики цветовой шкалы `[2, 3, 4, 6]`, пределы $10^4\ldots3\cdot10^7$,
подпись $m^{-1.83}$, ручная метка $2\cdot10^6$ — заменено на величины,
считаемые из самого прогона: при $\lambda=0$ неподвижная точка
$\alpha=-(3+\lambda)/2=-1.5$, и зашитые числа геометрии здесь просто уезжают
за пределы осей.

## Loading

The engine and the whole measurement layer come from `BF_analysis.py` in this folder, so nothing below is defined twice. Every figure is rebuilt from the stored run; no simulation is repeated.

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt

import BF_analysis as AN          # the whole measurement layer, engine included

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.figsize": (9, 3.2)})

FILES = {
    "constant": "runs/open_coagulation_conservative_constant.npz"
}

RUNS = {}
for tag, fn in FILES.items():
    if not os.path.exists(fn):
        raise FileNotFoundError(
            "%s is missing -- run the corresponding cell in the run-notebook next to "
            "this one; add_last_run writes it there on success." % fn)
    RUNS[tag] = AN.load(fn)
    print("loaded %-52s stop = %s" % (fn, RUNS[tag]["meta"].get("stop_reason")))

# AN.spectrum picks the estimator from meta: the averaged post-gate snapshot for an
# open run, the dt-weighted superposition for a closed one.  Both come back with an
# error bar attached.
SPEC = {tag: AN.spectrum(r) for tag, r in RUNS.items()}


# The growth-law plateau is computed here, once, from ALL the isochrones exactly as
# the engine recorded them.  Everything below reads it: the window that gets printed,
# the isochrones that get drawn and the exponent that gets quoted are then the same
# window by construction instead of by coincidence.  growth_compare also does the
# rebinning and refits it, so the check at the bottom of the notebook costs nothing
# extra here.
GROW = {tag: AN.growth_compare(r) for tag, r in RUNS.items() if "iso_counts" in r}


## What is in the file

Parameters, cost, the analysis stored at save time, and how large the counting error actually is — the empirical scatter beside the Poisson floor it can never go below.

In [ ]:
for tag, r in RUNS.items():
    print(AN.describe(r, tag))
    s = SPEC[tag]
    k = np.isfinite(s["F"]) & (s["F"] > 0)
    print("  %-16s %d snapshots averaged, K_eff median %.0f"
          % ("statistics", s["K"], np.nanmedian(s["K_eff"])))
    print("  %-16s empirical %.4f   Poisson floor %.4f   ratio %.2f"
          % ("median rel. error", np.nanmedian((s["sigma_emp"]/s["F"])[k]),
             np.nanmedian((s["sigma_pois"]/s["F"])[k]),
             np.nanmedian((s["sigma_emp"]/s["F"])[k])
             / max(np.nanmedian((s["sigma_pois"]/s["F"])[k]), 1e-30)))


## Evolution

The population must reach a plateau and the sink counter must climb steadily. $M_{\\rm sys}/m_{\\rm sink}$ is the number that decides whether a steady state is even possible.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for i, (tag, r) in enumerate(RUNS.items()):
    t = np.asarray(r["t"])
    ax[0].plot(t, r["live"], "-", lw=1.2, color="C%d" % i, label=tag)
    ax[1].plot(t, r["n_out"], "-", lw=1.2, color="C%d" % i, label=tag)
ax[0].set_xlabel("t"); ax[0].set_ylabel("live particles"); ax[0].set_ylim(bottom=0)
ax[0].set_title("(a) population -- must plateau")
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$n_{\rm out}$ absorbed at the sink")
ax[1].set_title("(b) the physical clock of the cascade")
for a in ax: a.legend(fontsize=7)
fig.tight_layout()

for tag, r in RUNS.items():
    msink = r["meta"]["sink_mass"] or np.inf
    print("%-14s live %d -> %d | sink = %d | M_out/M_in = %.3f | M_sys/m_sink = %.1f"
          % (tag, r["live"][0], r["live"][-1], float(r["sink_events"]),
             r["M_out"][-1]/max(r["M_in"][-1], 1), r["M_sys"][-1]/msink))
print("\nM_sys/m_sink below a few means one absorption empties the box: the run is a")
print("relaxation oscillator, not a steady state, and the spectrum is not measurable.")


## Where the growth law is a growth law

Every age bin is first turned into $\langle m\rangle(\tau)$ by the estimator that was
already here, untouched, and only then is the window chosen. The window is the plateau
of $d\log\langle m\rangle/d\log\tau$: the stretch of ages over which the isochrones are
self-similar, found automatically rather than set by hand. Its boundaries are printed
below before any picture is drawn, because a window that has to be seen to be believed
is not a measurement.

That same window then decides which isochrones appear on the spectrum panel. Age bins
below it are younger than one collision time and have not moved off the injection mass
— dozens of identical curves stacked in one place. Age bins above it are inside the
sink truncation. Neither is part of the cascade, and neither is drawn in colour.

Adjacent bins inside the window are merged until each curve carries enough counts to
be a curve. Merging is exact: the engine accumulates the age–mass histogram additively,
so summing neighbouring bins is precisely the histogram a coarser `iso_age_edges` would
have produced. Nothing is smoothed away that was ever measured; only age resolution is
spent, and it is spent at the top of the cascade where a single bin held a handful of
particles.

In [ ]:
# The window, stated as numbers, before it is used for anything.

for tag, r in RUNS.items():
    cmp = GROW.get(tag)
    if cmp is None:
        print("%-14s no isochrones stored in this run" % tag)
        continue
    b_th = r["meta"].get("analysis", {}).get("b_theory")
    cnt = np.asarray(r["iso_counts"]).sum(axis=1)
    print("=" * 88)
    print(AN.plateau_line(cmp["pl"], cmp["fit"], b_theory=b_th, tag=tag))
    g = cmp["groups"]
    isog = AN.iso_mean_mass(r, groups=g)
    print("  %d of %d age bins hold counts; %d of those are on the plateau,"
          " merged into %d isochrones"
          % (int((cnt > 0).sum()), cnt.size, int(cmp["k"].sum()), len(g)))
    print("  %-12s %12s %12s %13s %13s"
          % ("age bins", "tau_lo", "tau_hi", "<m>", "counts"))
    for grp, t0, t1, mb, S in zip(g, isog["tau_lo"], isog["tau_hi"],
                                  isog["mbar"], isog["counts"]):
        print("  %-12s %12.4g %12.4g %13.4g %13.4g"
              % ("%d-%d" % (grp[0], grp[-1]), t0, t1, mb, S))


## Spectrum

Compensated as $m^2\\,dN/dm$, with error bars, the plateau model and the theory line anchored to the same window so they differ in slope alone. The isochrones lie underneath; their sum must reproduce the measured spectrum, which is the completeness check.

In [ ]:
N_CURVES = 8      # <<-- target number of merged isochrones across the plateau

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.6 * n, 3.4), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a, s = axs[0][i], SPEC[tag]
    c = np.asarray(r["centers"])

    # The isochrones are chosen by the growth-law plateau printed above, not by a
    # count threshold, and adjacent age bins are merged until each curve has
    # something in it.  Everything rejected is still drawn, in grey underneath: the
    # window has to be visible against what it excludes, or the picture would be
    # confirming the window with the window.
    cmp = GROW.get(tag)
    idx, tau, ncand = AN.pick_isochrones(r, N_CURVES,
                                         pl=(cmp["pl"] if cmp else None))
    print("%-14s isochrones: %d age bins on the plateau out of %d in the grid"
          " -> %d merged curves" % (tag, ncand, len(tau), len(idx)))
    AN.draw_isochrones(a, r, idx, tau)
    if len(idx):
        tot = np.asarray(r["iso_dndm"]).sum(axis=0) / max(float(r["iso_snapshots"]), 1.0)
        a.loglog(c, np.where(tot > 0, tot * c**2, np.nan), "-", lw=3.5, alpha=.35,
                 color="0.4", zorder=2, label="sum of ALL isochrones")

    pl = AN.spectrum_plateau(r, spec=s)
    AN.plot_spectrum(a, r, spec=s, plateau=pl, label="measured $\\pm\\sigma$")

    at = r["meta"].get("analysis", {}).get("alpha_theory")
    if at and np.isfinite(pl["m_lo"]):
        xs = np.logspace(np.log10(pl["m_lo"]), np.log10(pl["m_hi"]), 30)
        A = AN.anchor_amplitude(c, s["F"], pl["m_lo"], pl["m_hi"], at)
        a.plot(xs, A * xs**at * xs**2, "--", lw=1.4, color="C1", zorder=5,
               label=r"theory $%.2f$" % at)

    w = AN.wls_powerlaw(c, s["F"], s["sigma"], mask=(c >= pl["m_lo"]) & (c <= pl["m_hi"]))
    print("      plateau alpha = %+.3f +- %.3f over %.2f dec | weighted fit %+.4f +- %.4f,"
          " chi2/dof = %.1f" % (pl["alpha"], pl["scatter"], pl["decades"],
                                w["p"], w["sigma_p"], w["chi2_dof"]))
    AN.compensated_ylim(a, c, s["F"])
    a.legend(fontsize=6, loc="lower left"); a.set_title(tag)
fig.tight_layout()
print("\nThe sum of the isochrones must reproduce the steady state -- that is the")
print("completeness check, and it uses every age bin, not the drawn subset.")
print("chi2/dof far above 1 means the deviations across the window are systematic,")
print("not statistical: the quoted sigma is then a lower bound, not the uncertainty.")


## Local slope

The diagnostic that decides the fitting window. A genuine inertial range is a plateau in $\\Gamma$; the contaminated ends are where it bends. The two shaded bands must overlap.

In [ ]:
HALF = 3        # half-width of the sliding window, in bins

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.2 * n, 2.9), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a, s = axs[0][i], SPEC[tag]
    c = np.asarray(r["centers"])
    an = r["meta"].get("analysis", {})
    mm, G = AN.local_slope(c, s["F"], half=HALF)
    a.semilogx(mm, G, "o-", ms=3, lw=.8)
    if an.get("alpha_theory") is not None:
        a.axhline(an["alpha_theory"], ls="--", lw=1, color="k",
                  label=r"theory $%.2f$" % an["alpha_theory"])
    if an.get("guard_lo") and np.isfinite(an["guard_lo"]):
        a.axvspan(an["guard_lo"], an["guard_hi"], alpha=.12, label="guard band")
    pl = AN.spectrum_plateau(r, spec=s)
    if np.isfinite(pl["m_lo"]):
        a.axvspan(pl["m_lo"], pl["m_hi"], alpha=.18, color="C1", label="auto plateau")
    a.set_ylim(-4, 1); a.set_xlabel("m")
    a.set_ylabel(r"$\Gamma = d\log F/d\log m$")
    a.legend(fontsize=6); a.set_title("local slope -- %s" % tag)
fig.tight_layout()


## Growth law from the isochrones

Every age bin is drawn with its error bar, the plateau is found automatically and the fit is weighted, so $\\sigma_b$ is propagated from the counts rather than assumed. The spread across windows is printed beside it, because at this precision the systematic is the larger of the two.

In [ ]:
n = len(RUNS)
fig, axs = plt.subplots(n, 2, figsize=(10.5, 3.4 * n), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    if "iso_counts" not in r:
        continue
    b_th = r["meta"].get("analysis", {}).get("b_theory")
    iso, pl, fit, k = AN.plot_growth(axs[i], r, b_theory=b_th, color="C%d" % i, tag=tag)

    print("=" * 80)
    print("%s   theory b = %s" % (tag, b_th))
    print("  plateau       b = %.3f +- %.3f over %.2f dec of tau, %d bins, <m> in [%.3g, %.3g]"
          % (pl["b"], pl["scatter"], pl["decades"], pl["n_bins"], pl["m_lo"], pl["m_hi"]))
    print("  weighted fit  b = %.4f +- %.4f (stat)   chi2/dof = %.2f   n = %d"
          % (fit["p"], fit["sigma_p"], fit["chi2_dof"], fit["n"]))
    if k.sum() == 0:
        # growth_plateau returning nothing is a VERDICT, not a crash: <m>(tau) is not
        # a power law over any stretch of this run.  Two honest reasons produce it.
        # Either the mean mass has saturated -- in an open fragmentation run that has
        # reached the sink, the number-weighted mean sits AT the sink and its slope is
        # zero, which the finder rejects as a shelf, correctly.  Or the slope swings
        # through a crossover, which is what the approach to the finite-time
        # singularity looks like when it is read in tau instead of (tau* - tau).
        # Read the local-slope panel beside the figure; the shape says which.
        print("  NO PLATEAU -- <m>(tau) is not a power law anywhere in this run.")
        print("  Flat at slope 0 in the right-hand panel = the mean mass has saturated")
        print("  at the sink.  A swing to large negative slope = the decay is a")
        print("  finite-time singularity and needs (tau* - tau), not tau.")
        continue
    print("  repeat correction n_rep over the plateau: %.1f ... %.1f"
          % (iso["n_rep"][k].min(), iso["n_rep"][k].max()))
    print("\n  b over reasonable windows -- the spread IS the systematic:")
    minj = float(r["meta"]["injection_mass"])
    msink = float(r["meta"]["sink_mass"] or np.inf)
    bs = []
    for C in (30, 100, 300, 1000):
        kc = (np.isfinite(iso["mbar"]) & (iso["counts"] > 1e3)
              & (iso["mbar"] > C * minj) & (iso["mbar"] < 0.3 * msink))
        f = AN.wls_powerlaw(iso["tau"], iso["mbar"], iso["sigma"], mask=kc)
        if np.isfinite(f["p"]):
            bs.append(f["p"])
            print("      cut %5g m_inj -> b = %.3f +- %.3f   chi2/dof = %7.2f  (%2d bins)"
                  % (C, f["p"], f["sigma_p"], f["chi2_dof"], f["n"]))
    if len(bs) > 1:
        print("  quote  b = %.2f +- %.2f (stat) +- %.2f (syst, window spread)"
              % (np.mean(bs), fit["sigma_p"], 0.5 * (max(bs) - min(bs))))
fig.tight_layout()


## Mass budget

Where the mass sits and whether it balances. Closed: $M_{\\rm sys}$ constant to machine precision. Open: $M_{\\rm out}/M_{\\rm in}\\to1$ is the definition of the steady state.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for i, (tag, r) in enumerate(RUNS.items()):
    t = np.asarray(r["t"])
    ax[0].plot(t, r["M_sys"], "-", lw=1.2, color="C%d" % i, label="%s: in the box" % tag)
    ax[0].plot(t, r["M_in"], "--", lw=1, color="C%d" % i, label="%s: injected" % tag)
    ax[0].plot(t, r["M_out"], ":", lw=1.4, color="C%d" % i, label="%s: absorbed" % tag)
    denom = np.where(np.asarray(r["M_in"]) > 0, r["M_in"], np.nan)
    ax[1].plot(t, np.asarray(r["M_out"]) / denom, "-", lw=1.2, color="C%d" % i, label=tag)
ax[0].set_xlabel("t"); ax[0].set_ylabel("mass"); ax[0].legend(fontsize=6)
ax[0].set_title("(a) where the mass is")
ax[1].axhline(1.0, ls="--", lw=1, color="k")
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$M_{\rm out}/M_{\rm in}$"); ax[1].legend(fontsize=7)
ax[1].set_title("(b) steady state means this reaches 1")
fig.tight_layout()

for tag, r in RUNS.items():
    print("%-14s M_sys %.6g -> %.6g | M_in %.4g | M_out %.4g | drift %+.2e"
          % (tag, r["M_sys"][0], r["M_sys"][-1], r["M_in"][-1], r["M_out"][-1],
             float(r["mass_drift"])))


## Evolution of the spectrum

Selected snapshots, compensated.

In [ ]:
N_SNAP = 8        # <<-- how many snapshots to draw per run

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.4 * n, 3.4), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a = axs[0][i]
    c = np.asarray(r["centers"]); D = np.asarray(r["dndm"]); t = np.asarray(r["t"])
    keep = [k for k in range(D.shape[0]) if np.any(D[k] > 0)]
    sel = np.unique(np.linspace(0, len(keep) - 1, min(N_SNAP, len(keep))).astype(int))
    cm = plt.cm.plasma(np.linspace(0, .88, len(sel)))
    for col, sidx in zip(cm, sel):
        kk = keep[sidx]
        a.loglog(c, np.where(D[kk] > 0, D[kk] * c**2, np.nan), lw=1.1, color=col,
                 label="t = %.3g" % t[kk])
    a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
    a.legend(fontsize=5.5, ncol=2); a.set_title("evolution -- %s" % tag)
fig.tight_layout()


## The same growth law, sampled twice

The exponent is measured again on the merged age bins the isochrone panel was drawn
from, and plotted on top of the original fine-binned one. This is the check that the
widening is honest rather than flattering.

Counts add exactly, so a merged bin is the bin a coarser age grid would have recorded.
If the window really is a single power law, resampling it in age cannot move the
slope, and the two fits must agree. Where they do not, the disagreement is information:
the wide bins are straddling curvature, which means the window is not a power law over
its whole length whatever the fitted $\sigma_b$ says. Read the difference together with
$\chi^2/\rm dof$ — they fail together, and for the same reason.

In [ ]:
n = len(GROW)
if n == 0:
    print("no isochrones in this run -- nothing to compare")
else:
    fig, axs = plt.subplots(n, 2, figsize=(10.5, 3.4 * n), squeeze=False)
    for i, (tag, cmp) in enumerate(GROW.items()):
        r = RUNS[tag]
        b_th = r["meta"].get("analysis", {}).get("b_theory")
        AN.plot_growth_compare(axs[i], r, cmp=cmp, b_theory=b_th,
                               color="C%d" % i, tag=tag)
        f, fg = cmp["fit"], cmp["fit_g"]
        d = f["p"] - fg["p"]
        sd = np.hypot(f["sigma_p"], fg["sigma_p"])
        print("=" * 88)
        print("%s   theory b = %s" % (tag, b_th))
        print("  original bins   b = %+.4f +- %.4f   chi2/dof = %8.2f   n = %2d"
              % (f["p"], f["sigma_p"], f["chi2_dof"], f["n"]))
        print("  rebinned        b = %+.4f +- %.4f   chi2/dof = %8.2f   n = %2d"
              % (fg["p"], fg["sigma_p"], fg["chi2_dof"], fg["n"]))
        print("  difference        %+.4f   =  %.1f sigma"
              % (d, abs(d) / sd if sd > 0 else np.nan))
        if sd > 0 and abs(d) / sd > 3:
            print("  -> the merging is exact, so this is curvature inside the window,")
            print("     not an artefact of widening the bins.  Narrow the window or")
            print("     accept that b is a local slope here rather than an exponent.")
        else:
            print("  -> consistent: resampling the window in age does not move the slope,")
            print("     which is what a genuine power law is supposed to do.")
    fig.tight_layout()


---
## PRL dn/dm spectrum

Сырой спектр и отобранные слитые изохроны; старшая из них подогнана
логнормальным распределением. Нижняя панель — локальный наклон, чтобы
показатель читался с рисунка, а не только с подписи.

`FIT_GROUP = -2` — какую изохрону подгонять, считая с конца. Самая последняя
обычно уже обрезана стоком.

In [ ]:
# PRL_STYLE_DNDM_SPECTRUM
from matplotlib.colors import LogNorm

TAG = "constant"
r = RUNS[TAG]
s = SPEC[TAG]
cmp = GROW.get(TAG)
pl_for_iso = cmp["pl"] if cmp is not None else None
idx, tau, ncand = AN.pick_isochrones(r, N_CURVES, pl=pl_for_iso)

groups = [np.atleast_1d(np.asarray(g, int)) for g in idx]
if len(groups) == 0:
    raise RuntimeError("No rebinned isochrones were selected for the PRL dn/dm plot.")

m = np.asarray(r["centers"], float)
counts = np.asarray(r["iso_counts"], float)
iso_dndm = np.asarray(r["iso_dndm"], float)
nsnap_iso = max(float(r["iso_snapshots"]), 1.0)
analysis = r["meta"].get("analysis", {})
m_inj = float(analysis.get("m_inj", 1.0))
m_sink = float(analysis.get("m_sink", np.nan))

age = np.array([
    np.average(tau[g], weights=np.maximum(counts[g].sum(axis=1), 1e-300))
    for g in groups
])
order = np.argsort(age)
groups = [groups[i] for i in order]
age = age[order]
AGE_UNIT = 1e-6
age_scaled = age / AGE_UNIT
FIT_GROUP = -2    # one from the end; use -1 for the very last isochrone
old_group = groups[FIT_GROUP]

# Fit the selected rebinned isochrone as dN/dm = A/(m sigma sqrt(2 pi)) exp[-(ln m - mu)^2/(2 sigma^2)].
y_old = iso_dndm[old_group].sum(axis=0) / nsnap_iso
cnt_old = counts[old_group].sum(axis=0)
fit_mask = np.isfinite(m) & np.isfinite(y_old) & (m > 0) & (y_old > 0) & (cnt_old >= 3.0)
if fit_mask.sum() < 3:
    raise RuntimeError("The selected rebinned isochrone has fewer than three usable bins for a lognormal fit.")

x_fit = m[fit_mask]
y_fit = y_old[fit_mask]
try:
    widths = np.asarray(r["widths"], float)[fit_mask]
except Exception:
    widths = np.gradient(x_fit)
weights = np.maximum(y_fit * widths, 0.0)
if not np.any(weights > 0):
    weights = y_fit / np.nanmax(y_fit)
logx = np.log(x_fit)
mu0 = float(np.average(logx, weights=weights))
sigma0 = float(np.sqrt(np.average((logx - mu0) ** 2, weights=weights)))
sigma0 = max(sigma0, 0.15)
area0 = float(np.sum(y_fit * widths))

def lognormal_dndm(x, area, mu, sigma):
    x = np.asarray(x, float)
    sigma = np.maximum(sigma, 1e-12)
    return area / (x * sigma * np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * ((np.log(x) - mu) / sigma) ** 2)

fit_method = "moment estimate"
try:
    from scipy.optimize import curve_fit
    popt, pcov = curve_fit(
        lognormal_dndm,
        x_fit,
        y_fit,
        p0=(area0, mu0, sigma0),
        bounds=([0.0, np.log(x_fit.min()) - 5.0, 1e-3],
                [np.inf, np.log(x_fit.max()) + 5.0, 10.0]),
        maxfev=20000,
    )
    area_fit, mu_fit, sigma_fit = [float(v) for v in popt]
    fit_method = "nonlinear least squares"
except Exception as exc:
    area_fit, mu_fit, sigma_fit = area0, mu0, sigma0
    print("SciPy lognormal fit was not available; using moment estimate:", exc)

plot_xlim = (0.5 * m_inj, m_sink * 1.5) if np.isfinite(m_sink) else (x_fit.min(), x_fit.max())
x_line_for_limits = np.logspace(np.log10(x_fit.min()), np.log10(x_fit.max()), 300)
y_line_for_limits = lognormal_dndm(x_line_for_limits, area_fit, mu_fit, sigma_fit)
x_line = np.logspace(np.log10(plot_xlim[0]), np.log10(plot_xlim[1]), 600)
y_line = lognormal_dndm(x_line, area_fit, mu_fit, sigma_fit)

prl_rc = {
    "figure.dpi": 160,
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 6.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.grid": False,
}

with plt.rc_context(prl_rc):
    fig = plt.figure(figsize=(3.35, 3.05))
    gs = fig.add_gridspec(
        2, 2,
        width_ratios=[1.0, 0.045],
        height_ratios=[3.25, 0.75],
        wspace=0.07,
        hspace=0.0,
    )
    ax = fig.add_subplot(gs[0, 0])
    ax_slope = fig.add_subplot(gs[1, 0], sharex=ax)
    cax = fig.add_subplot(gs[0, 1])

    valid_spec = np.isfinite(s["F"]) & (s["F"] > 0)
    ax.errorbar(
        m[valid_spec], s["F"][valid_spec], yerr=s["sigma"][valid_spec],
        fmt="o", ms=2.0, mfc="white", mec="0.25", mew=0.5,
        ecolor="0.75", elinewidth=0.45, capsize=1.0,
        color="0.25", zorder=2, label=r"spectrum"
    )

    norm = LogNorm(vmin=max(age_scaled.min(), np.nextafter(0, 1)),
                   vmax=max(age_scaled.max(), age_scaled.min() * 1.0001))
    cmap = plt.cm.viridis
    for g, tk in zip(groups, age_scaled):
        y = iso_dndm[g].sum(axis=0) / nsnap_iso
        cg = counts[g].sum(axis=0)
        ok = np.isfinite(y) & (y > 0) & (cg >= 3.0)
        ax.loglog(
            m[ok], y[ok], "o", ms=2.6, markeredgewidth=0.0,
            color=cmap(norm(tk)), alpha=0.92, zorder=3
        )

    ax.loglog(x_line, y_line, "-", color="k", lw=1.4, zorder=5,
              label="lognormal fit")
    fit_peak = int(np.nanargmax(y_line))
    ax.text(
        x_line[fit_peak], y_line[fit_peak] * 10.0 ** 1.5,
        "Lognormal\n$\\downarrow$",
        ha="center", va="bottom", fontsize=7, color="0.1", linespacing=0.9,
    )
    ax.axvline(m_inj*0.9, color="0.1", lw=0.8, ls="--", zorder=1, label="_nolegend_")
    ax.text(m_inj * 1.12, 0.96, "Injection", transform=ax.get_xaxis_transform(),
            ha="left", va="top", fontsize=7, color="0.1")
    if np.isfinite(m_sink):
        ax.axvline(m_sink, color="0.1", lw=0.8, ls="--", zorder=1, label="_nolegend_")
        ax.text(m_sink / 1.12, 0.96, "Sink", transform=ax.get_xaxis_transform(),
                ha="right", va="top", fontsize=7, color="0.1")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    #  Тики -- сами возрасты выбранных изохрон, прорежённые, а не зашитый список
    #  [2, 3, 4, 6]: он подобран под геометрический прогон и на постоянном ядре
    #  уезжает за пределы шкалы, оставляя цветовую полосу без единой подписи.
    _la = np.log10(age_scaled)
    _keep = [0]
    for _i in range(1, _la.size):
        if (_la[_i] - _la[_keep[-1]]) > 0.12 * (_la.max() - _la.min() + 1e-12):
            _keep.append(_i)
    if _keep[-1] != _la.size - 1:
        _keep[-1] = _la.size - 1
    cb.set_ticks(age_scaled[_keep])
    cb.set_ticklabels(["%.3g" % a for a in age_scaled[_keep]])
    cb.minorticks_off()
    cb.set_label(r"age $\tau$  ($10^{%d}$)" % round(np.log10(AGE_UNIT)),
                 labelpad=2, fontsize=7)
    cb.ax.tick_params(direction="in", length=2.5, width=0.7)

    y_all = [s["F"][valid_spec], y_line_for_limits]
    for g in groups:
        y = iso_dndm[g].sum(axis=0) / nsnap_iso
        cg = counts[g].sum(axis=0)
        ok = np.isfinite(y) & (y > 0) & (cg >= 3.0)
        if np.any(ok):
            y_all.append(y[ok])
    y_pos = np.concatenate([yy[np.isfinite(yy) & (yy > 0)] for yy in y_all])
    ax.set_xlim(*plot_xlim)
    ax.set_ylim(y_pos.min() / 2.0, y_pos.max() * 2.5)
    ax.set_ylabel(r"$dN/dm$")
    ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

    mm, local_slope = AN.local_slope(m, s["F"], half=HALF)
    ok_slope = np.isfinite(mm) & np.isfinite(local_slope)
    ax_slope.semilogx(
        mm[ok_slope], local_slope[ok_slope], "o",
        ms=2.2, mfc="white", mec="0.25", mew=0.5,
        color="0.25", zorder=3,
    )
    alpha_theory = analysis.get("alpha_theory")
    if alpha_theory is not None and np.isfinite(alpha_theory):
        ax_slope.axhline(alpha_theory, color="0.1", lw=0.8, ls="--", zorder=2)
        ax_slope.text(0.88, alpha_theory, "%.2f" % alpha_theory,
                      transform=ax_slope.get_yaxis_transform(), ha="left", va="center",
                      fontsize=7, color="0.1",
                      bbox=dict(facecolor="white", edgecolor="none", pad=0.3, alpha=0.85))
        ax_slope.text(0.95, alpha_theory - 0.06, "theory", transform=ax_slope.get_yaxis_transform(),
                      ha="right", va="top", fontsize=7, color="0.1")
    ax_slope.axvline(m_inj*0.9, color="0.1", lw=0.8, ls="--", zorder=1)
    if np.isfinite(m_sink):
        ax_slope.axvline(m_sink, color="0.1", lw=0.8, ls="--", zorder=1)
    #  Раньше окно наклона было зашито как (-2.5, -1.5) -- под alpha = -1.83
    #  геометрического прогона.  На постоянном ядре неподвижная точка -1.5
    #  оказывается ровно на краю, и линия теории прилипает к рамке.  Центр
    #  берётся из теории, если она записана, иначе из самих данных.
    _c = alpha_theory if (alpha_theory is not None and np.isfinite(alpha_theory)) \
         else float(np.nanmedian(local_slope[ok_slope]))
    ax_slope.set_xlim(*plot_xlim)
    ax_slope.set_ylim(_c - 0.55, _c + 0.55)
    ax_slope.set_xlabel(r"$m$")
    ax_slope.set_ylabel("slope", fontsize=7)
    # ax.set_title("geometric", pad=2)
    # ax.legend(frameon=False, loc="lower left", handlelength=1.6, borderpad=0.2)
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.14, top=0.98, hspace=0.0, wspace=0.07)

print("fitted rebinned isochrone: tau = %.4g, bins = %d, fit = %s" %
      (age[FIT_GROUP], fit_mask.sum(), fit_method))
print("lognormal parameters: area = %.4g, mu = %.4g, sigma = %.4g, median m = %.4g" %
      (area_fit, mu_fit, sigma_fit, np.exp(mu_fit)))


---
## PRL compensated spectrum

Однопанельная PRL-версия итогового спектра, компенсированного как
$m^2\,dN/dm$. Показатель в подписи — измеренное плато, а не зашитое число.

In [ ]:
# PRL_STYLE_COMPENSATED_SPECTRUM
from matplotlib.colors import LogNorm

TAG = "constant"
r = RUNS[TAG]
s = SPEC[TAG]
cmp = GROW.get(TAG)
pl_for_iso = cmp["pl"] if cmp is not None else None
idx, tau, ncand = AN.pick_isochrones(r, N_CURVES, pl=pl_for_iso)

groups = [np.atleast_1d(np.asarray(g, int)) for g in idx]
if len(groups) == 0:
    raise RuntimeError("No rebinned isochrones were selected for the compensated PRL plot.")

m = np.asarray(r["centers"], float)
counts = np.asarray(r["iso_counts"], float)
iso_dndm = np.asarray(r["iso_dndm"], float)
nsnap_iso = max(float(r["iso_snapshots"]), 1.0)
analysis = r["meta"].get("analysis", {})
m_inj = float(analysis.get("m_inj", 1.0))
m_sink = float(analysis.get("m_sink", np.nan))
plot_xlim = (0.5 * m_inj, m_sink * 1.5) if np.isfinite(m_sink) else (m[m > 0].min(), m.max())

age = np.array([
    np.average(tau[g], weights=np.maximum(counts[g].sum(axis=1), 1e-300))
    for g in groups
])
order = np.argsort(age)
groups = [groups[i] for i in order]
age = age[order]
AGE_UNIT = 1e-6
age_scaled = age / AGE_UNIT
FIT_GROUP = -2    # one from the end; use -1 for the very last isochrone
old_group = groups[FIT_GROUP]

# Fit the selected rebinned isochrone in raw dN/dm, then compensate only for display.
y_old = iso_dndm[old_group].sum(axis=0) / nsnap_iso
cnt_old = counts[old_group].sum(axis=0)
fit_mask = np.isfinite(m) & np.isfinite(y_old) & (m > 0) & (y_old > 0) & (cnt_old >= 3.0)
if fit_mask.sum() < 3:
    raise RuntimeError("The selected rebinned isochrone has fewer than three usable bins for a lognormal fit.")

x_fit = m[fit_mask]
y_fit = y_old[fit_mask]
try:
    widths = np.asarray(r["widths"], float)[fit_mask]
except Exception:
    widths = np.gradient(x_fit)
weights = np.maximum(y_fit * widths, 0.0)
if not np.any(weights > 0):
    weights = y_fit / np.nanmax(y_fit)
logx = np.log(x_fit)
mu0 = float(np.average(logx, weights=weights))
sigma0 = float(np.sqrt(np.average((logx - mu0) ** 2, weights=weights)))
sigma0 = max(sigma0, 0.15)
area0 = float(np.sum(y_fit * widths))

def lognormal_dndm(x, area, mu, sigma):
    x = np.asarray(x, float)
    sigma = np.maximum(sigma, 1e-12)
    return area / (x * sigma * np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * ((np.log(x) - mu) / sigma) ** 2)

fit_method = "moment estimate"
try:
    from scipy.optimize import curve_fit
    popt, pcov = curve_fit(
        lognormal_dndm,
        x_fit,
        y_fit,
        p0=(area0, mu0, sigma0),
        bounds=([0.0, np.log(x_fit.min()) - 5.0, 1e-3],
                [np.inf, np.log(x_fit.max()) + 5.0, 10.0]),
        maxfev=20000,
    )
    area_fit, mu_fit, sigma_fit = [float(v) for v in popt]
    fit_method = "nonlinear least squares"
except Exception as exc:
    area_fit, mu_fit, sigma_fit = area0, mu0, sigma0
    print("SciPy lognormal fit was not available; using moment estimate:", exc)

x_line_for_limits = np.logspace(np.log10(x_fit.min()), np.log10(x_fit.max()), 300)
y_line_for_limits = lognormal_dndm(x_line_for_limits, area_fit, mu_fit, sigma_fit)
x_line = np.logspace(np.log10(plot_xlim[0]), np.log10(plot_xlim[1]), 600)
y_line = lognormal_dndm(x_line, area_fit, mu_fit, sigma_fit)
y_line_comp = y_line * x_line**2

prl_rc = {
    "figure.dpi": 160,
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 6.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.grid": False,
}

with plt.rc_context(prl_rc):
    fig = plt.figure(figsize=(3.35, 2.45))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 0.045], wspace=0.07)
    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[0, 1])

    valid_spec = np.isfinite(s["F"]) & (s["F"] > 0) & np.isfinite(m) & (m > 0)
    ax.errorbar(
        m[valid_spec], (s["F"] * m**2)[valid_spec], yerr=(s["sigma"] * m**2)[valid_spec],
        fmt="o", ms=2.0, mfc="white", mec="0.25", mew=0.5,
        ecolor="0.75", elinewidth=0.45, capsize=1.0,
        color="0.25", zorder=2, label=r"spectrum"
    )

    norm = LogNorm(vmin=max(age_scaled.min(), np.nextafter(0, 1)),
                   vmax=max(age_scaled.max(), age_scaled.min() * 1.0001))
    cmap = plt.cm.viridis
    for g, tk in zip(groups, age_scaled):
        y = iso_dndm[g].sum(axis=0) / nsnap_iso
        cg = counts[g].sum(axis=0)
        ok = np.isfinite(y) & (y > 0) & (cg >= 3.0) & np.isfinite(m) & (m > 0)
        ax.loglog(
            m[ok], (y * m**2)[ok], "o", ms=2.6, markeredgewidth=0.0,
            color=cmap(norm(tk)), alpha=0.92, zorder=3
        )

    ax.loglog(x_line, y_line_comp, "-", color="k", lw=1.4, zorder=1.5,
              label="lognormal fit")
    #  Подпись подгонки ставится ПОСЛЕ того, как выставлены пределы: её высота
    #  была прижата к 1.2e7 -- потолку геометрического рисунка, -- и на любом
    #  другом прогоне надпись уезжала за рамку или садилась на "Sink".
    fit_peak = int(np.nanargmax(y_line_comp))

    ax.axvline(m_inj*0.9, color="0.1", lw=0.8, ls="--", zorder=1, label="_nolegend_")
    ax.text(m_inj * 1.12, 0.96, "Injection", transform=ax.get_xaxis_transform(),
            ha="left", va="top", fontsize=7, color="0.1")
    if np.isfinite(m_sink):
        ax.axvline(m_sink, color="0.1", lw=0.8, ls="--", zorder=1, label="_nolegend_")
        ax.text(m_sink / 1.12, 0.96, "Sink", transform=ax.get_xaxis_transform(),
                ha="right", va="top", fontsize=7, color="0.1")
    #  Показатель берётся ИЗМЕРЕННЫЙ, а не зашитый -1.83 геометрического прогона:
    #  подписывать чужое число на своём рисунке -- ровно тот способ ошибиться,
    #  который никто не заметит, потому что цифра выглядит правдоподобно.
    _pl = AN.spectrum_plateau(r, spec=s)
    ax.text(0.50, 0.6, r"$\frac{dN}{dm}\propto m^{%.2f}$" % _pl["alpha"],
            transform=ax.transAxes, ha="center", va="center", fontsize=8, color="0.1")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    #  Тики -- сами возрасты выбранных изохрон, прорежённые, а не зашитый список
    #  [2, 3, 4, 6]: он подобран под геометрический прогон и на постоянном ядре
    #  уезжает за пределы шкалы, оставляя цветовую полосу без единой подписи.
    _la = np.log10(age_scaled)
    _keep = [0]
    for _i in range(1, _la.size):
        if (_la[_i] - _la[_keep[-1]]) > 0.12 * (_la.max() - _la.min() + 1e-12):
            _keep.append(_i)
    if _keep[-1] != _la.size - 1:
        _keep[-1] = _la.size - 1
    cb.set_ticks(age_scaled[_keep])
    cb.set_ticklabels(["%.3g" % a for a in age_scaled[_keep]])
    cb.minorticks_off()
    cb.set_label(r"age $\tau$  ($10^{%d}$)" % round(np.log10(AGE_UNIT)),
                 labelpad=2, fontsize=7)
    cb.ax.tick_params(direction="in", length=2.5, width=0.7)

    y_all = [(s["F"] * m**2)[valid_spec], y_line_for_limits * x_line_for_limits**2]
    for g in groups:
        y = iso_dndm[g].sum(axis=0) / nsnap_iso
        cg = counts[g].sum(axis=0)
        ok = np.isfinite(y) & (y > 0) & (cg >= 3.0) & np.isfinite(m) & (m > 0)
        if np.any(ok):
            y_all.append((y * m**2)[ok])
    y_pos = np.concatenate([yy[np.isfinite(yy) & (yy > 0)] for yy in y_all])
    #  Пределы -- из данных.  Зашитые (1e4, 3e7) и ручная метка 2*10^6 -- числа
    #  геометрического прогона; при другой нормировке коробки они срезают кривые.
    #  Вниз не больше 3.5 декад от максимума, иначе рисунок вырождается в поле.
    ax.set_xlim(*plot_xlim)
    _yhi = 10.0 ** np.ceil(np.log10(y_pos.max() * 2.5))
    _ylo = 10.0 ** np.floor(np.log10(max(y_pos.min() / 1.5,
                                         y_pos.max() / 10.0 ** 3.5)))
    ax.set_ylim(_ylo, _yhi)
    ax.text(x_line[fit_peak] * 0.55,
            min(y_line_comp[fit_peak] * 10.0 ** 0.35, _yhi / 10.0 ** 0.8),
            "Lognormal\nfit\n$\\downarrow$",
            ha="center", va="bottom", fontsize=7, color="0.1", linespacing=0.9)
    ax.set_xlabel(r"$m$")
    ax.set_ylabel(r"$m^2\,dN/dm$")
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.16, top=0.98, wspace=0.07)

print("fitted rebinned isochrone: tau = %.4g, bins = %d, fit = %s" %
      (age[FIT_GROUP], fit_mask.sum(), fit_method))
print("lognormal parameters: area = %.4g, mu = %.4g, sigma = %.4g, median m = %.4g" %
      (area_fit, mu_fit, sigma_fit, np.exp(mu_fit)))
